In [ ]:
%cd ..

# Student Health Risk — Estimating the Score Ceiling

**What is the highest balanced accuracy we can possibly achieve on this dataset?**

This notebook decomposes the gap between a perfect 1.0 and our current best submission
into irreducible components — label noise, missingness, and recoverable signal.

**Key finding**: The label-generating rule depends on only 3 features:
`sleep_duration`, `stress_level`, and `physical_activity_level`. Two of the three
have recoverable substitutes via proxy features, but `stress_level` is independent
noise — when missing (12% of rows), we cannot distinguish `fit` from `unhealthy`.

The realistic ceiling is **~0.951**, meaning there is ~0.044 of headroom from our
current ~0.907 OOF, but ~0.033 of that is **irreducible label noise**. The true
exploitable gap is only ~0.011.

Based on [nybbler's analysis](https://www.kaggle.com/code/nybbler/s6e7-estimating-the-score-ceiling).

## 1. Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from lightgbm import LGBMClassifier, log_evaluation
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

from src.student_health.features import CAT_COLS, NUM_COLS, TARGET_COL

load_dotenv()
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11, "axes.grid": True,
    "axes.axisbelow": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False,
})
C = {
    "atrisk": "#5B8DEF", "fit": "#2CA58D", "unhealthy": "#E15759",
    "accent": "#9B5DE5", "grey": "#9AA0A6",
}

# ── Data paths ──
_DATA_CANDIDATES = [Path("data/raw"), Path("../data/raw")]
DATA_DIR = next((p for p in _DATA_CANDIDATES if (p / "train.csv").exists()), _DATA_CANDIDATES[0])

# Original source dataset (optional — used only for rule verification)
ORIG_DIR = None
try:
    _orig = kagglehub.dataset_download("ziya07/college-student-health-behavior-dataset")
    ORIG_DIR = Path(_orig)
    print(f"✓ Original dataset: {ORIG_DIR}")
except Exception:
    print("⚠ Original dataset not available — rule verification skipped")

RANDOM_STATE = 42
N_FOLDS = 5
TARGET_LABELS = ["fit", "at-risk", "unhealthy"]

## 2. Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_ids = test["id"].copy()

orig = None
enh = None
if ORIG_DIR is not None:
    orig = pd.read_csv(ORIG_DIR / "student_health_dataset_50k.csv")
    enh = pd.read_csv(ORIG_DIR / "enhanced_student_health_dataset_50k.xls")  # actually a CSV

print(f"Train: {train.shape}  |  Test: {test.shape}")
if orig is not None:
    print(f"Original (source rule): {orig.shape}")
    print(f"Original (enhanced):    {enh.shape}")
print(f"\nTarget distribution:\n{train[TARGET].value_counts(normalize=True).mul(100).round(2)}")

## 3. The Label-Generating Rule

[broccoli beef (discussion 717222)](https://www.kaggle.com/competitions/playground-series-s6e7/discussion/717222)
reverse-engineered the generating rule. It is a depth-4 decision tree on just three features:
`sleep_duration`, `stress_level`, `physical_activity_level`.

We verify the rule achieves **100% accuracy on the original dataset** and ~99% on
competition rows where all three features are present (the ~1% gap is injected label noise).

In [ ]:
RULE = ["sleep_duration", "stress_level", "physical_activity_level"]


def generating_rule(df):
    s = df["sleep_duration"]
    st = df["stress_level"]
    a = df["physical_activity_level"]
    o = pd.Series(index=df.index, dtype="object")

    o[(s < 6) & (st == "high")] = "unhealthy"
    o[(s < 6) & (st != "high")] = "at-risk"

    ge = s >= 6
    o[ge & (st != "low")] = "at-risk"

    lo = ge & (st == "low")
    o[lo & (a != "active")] = "at-risk"
    o[lo & (a == "active")] = "at-risk"  # intermediate group
    o[lo & (a == "active") & (s >= 7)] = "fit"
    return o


# Verify on original data
if orig is not None:
    acc_orig = (generating_rule(orig) == orig[TARGET]).mean()
    print(f"Rule accuracy on original (complete): {acc_orig:.4f}")

# Verify on competition train — only rows where all rule features exist
mask = train[RULE].notna().all(axis=1)
acc_comp = (generating_rule(train[mask]) == train.loc[mask, TARGET]).mean()
pct_complete = mask.mean() * 100
print(f"Rule accuracy on competition train (complete rows, {pct_complete:.0f}%): {acc_comp:.4f}")
print()
print("On the ~74% of rows where all three rule features are present,")
print("the problem is essentially solved by a hand-written rule.")
print("**The entire competition is decided by the other ~26%** — rows where")
print("a rule feature is missing and must be inferred.")

## 4. stress_level Is Independent Noise

The original dataset contains five additional columns that were dropped from the
competition — including *academic_pressure* and *mental_health_status*. If stress
were predictable, these would be the strongest predictors. Let's check.

In [ ]:
if enh is not None:
    # Only use columns present in both datasets
    comp_feats = [c for c in NUM_COLS + CAT_COLS if c in enh.columns]
    dropped = ["screen_time", "sitting_time", "mental_health_status",
               "academic_pressure", "social_relationships",
               "smoking_status", "alcohol_intake"]

    y_stress = pd.Categorical(enh["stress_level"])
    yc = y_stress.codes
    base_acc = pd.Series(yc).value_counts(normalize=True).max()

    def predict_stress(cols, label):
        X = enh[cols].copy()
        for c in X.columns:
            if not pd.api.types.is_numeric_dtype(X[c]):
                X[c] = X[c].astype("category")
        Xtr, Xva, ytr, yva = train_test_split(
            X, yc, test_size=0.25, random_state=42, stratify=yc
        )
        m = LGBMClassifier(
            objective="multiclass", num_class=3, n_estimators=300,
            learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
            verbose=-1, n_jobs=-1,
        )
        m.fit(Xtr, ytr)
        p = m.predict(Xva)
        ba = balanced_accuracy_score(yva, p)
        print(f"{label:34s} BA={ba:.3f}  acc={accuracy_score(yva,p):.3f}")
        return ba, dict(zip(cols, m.feature_importances_))

    print(f"(random 3-class BA = 0.333 | majority-class acc = {base_acc:.3f})")
    ba_comp, _ = predict_stress(comp_feats, "comp-available features")
    ba_all, imp = predict_stress(
        comp_feats + dropped, "comp features + 7 DROPPED columns"
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    axes[0].axhline(1 / 3, ls="--", color=C["grey"], label="random (0.333)")
    axes[0].bar(
        ["comp\nfeatures", "comp +\ndropped cols"],
        [ba_comp, ba_all],
        color=[C["atrisk"], C["accent"]], width=0.6,
    )
    axes[0].set_ylim(0, 0.6)
    axes[0].set_ylabel("balanced accuracy predicting stress")
    axes[0].set_title("Stress is unpredictable — even WITH dropped psych columns")
    axes[0].legend(frameon=False)
    for i, v in enumerate([ba_comp, ba_all]):
        axes[0].text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)

    items = sorted(imp.items(), key=lambda x: -x[1])
    colors_bar = [
        C["accent"] if k in dropped else C["grey"] for k, _ in items
    ][::-1]
    axes[1].barh(
        [k for k, _ in items][::-1],
        [v for _, v in items][::-1],
        color=colors_bar,
    )
    axes[1].set_title("All importances ~equal → model finds no signal")
    axes[1].set_xlabel("LightGBM importance")
    plt.tight_layout()
    plt.show()
else:
    print("Enhanced original data not available — skipping stress analysis.")

**Verdict.** In the source data, `stress_level` is statistically independent of every
other feature — lifestyle, physiological, and psychological. It was assigned as an
independent random draw with fixed marginals (~medium 40 / low 30 / high 30).

This is the **critical insight for the ceiling**: when `stress_level` is missing
(12% of competition rows), we cannot distinguish `fit` from `unhealthy` because
those two classes are separated only by `stress_level` (low stress ⇒ `fit`,
high stress ⇒ `unhealthy`).

## 5. Decomposing the Score Ceiling

We compute the optimal balanced accuracy under three scenarios:

1. **Label-noise ceiling** — what BA would an oracle achieve if it knew the true
   generating rule *and* had complete data? This is limited only by the ~1%
   injected label noise.
2. **Naive missingness floor** — what BA does the optimal rule achieve on the
   actual competition data? This accounts for missing rule features.
3. **Proxy recovery** — how much can a good model claw back via correlated features
   (e.g. `step_count` → activity level, `sleep_quality` → sleep bin)?

In [ ]:
def _sbin(s):
    return np.where(
        s.isna(), "NA",
        np.where(s < 6, "<6", np.where(s < 7, "6-7", ">=7"))
    )


def optimal_ba(df):
    # Optimal balanced accuracy via class-balanced argmax per feature group
    cl = ["at-risk", "fit", "unhealthy"]
    sig = [
        "|".join(map(str, t))
        for t in zip(
            _sbin(df["sleep_duration"]),
            df["stress_level"].fillna("NA"),
            df["physical_activity_level"].fillna("NA"),
        )
    ]
    yy = df[TARGET].values
    Nk = {c: int((yy == c).sum()) for c in cl}
    counts = (
        pd.DataFrame({"sig": sig, "y": yy})
        .groupby("sig")["y"]
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=cl, fill_value=0)
    )
    pred = (counts / pd.Series(Nk)).idxmax(axis=1)
    correct = {c: 0 for c in cl}
    for g, pc in pred.items():
        correct[pc] += int(counts.loc[g, pc])
    return float(np.mean([correct[c] / Nk[c] for c in cl]))


# ── Scenario ceilings ──
noise_ceiling = optimal_ba(train[train[RULE].notna().all(axis=1)])
naive_floor = optimal_ba(train)
achieved = 0.950  # top public LB at time of writing → realistic ceiling
our_best_cv = 0.907  # our LightGBM baseline OOF

label_noise = 1.0 - noise_ceiling
miss_cost = noise_ceiling - naive_floor
proxy_recovery = achieved - naive_floor
our_gap_to_ceiling = achieved - our_best_cv

print("=" * 60)
print("  SCORE CEILING DECOMPOSITION")
print("=" * 60)
print(f"  Perfect score:                  1.0000")
print(f"  Label-noise ceiling:            {noise_ceiling:.4f}")
print(f"  Naive missingness floor:        {naive_floor:.4f}")
print(f"  Top public LB (realistic ceil): {achieved:.4f}")
print(f"  Our best OOF:                   {our_best_cv:.4f}")
print()
print(f"  Label noise (irreducible):      -{label_noise:.4f}")
print(f"  Missingness cost (recoverable): -{miss_cost:.4f}")
print(f"  Proxy recovery potential:       +{proxy_recovery:.4f}")
print(f"  Our gap to ceiling:             {our_gap_to_ceiling:.4f}")
print(f"  --- of which is irreducible:    {label_noise:.4f}")
print(f"  --- truly exploitable gap:      {our_gap_to_ceiling - label_noise:.4f}")

### 5.1  Waterfall: Perfect → Realistic Ceiling

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.9))

# Build bars
ax.bar(0, 1.0, color="#CFCFCF", width=0.6)

# Label noise drop
ax.bar(1, label_noise, bottom=noise_ceiling, color=C["unhealthy"], width=0.6)

# Missingness drop
ax.bar(2, miss_cost, bottom=naive_floor, color=C["accent"], width=0.6)

# Proxy recovery
ax.bar(3, proxy_recovery, bottom=naive_floor, color=C["fit"], width=0.6)

# Our submission
ax.bar(4, our_best_cv, color="#3A7CA5", width=0.6)

# Ceiling line
ax.axhline(achieved, ls="--", lw=1, color="#444")
ax.text(
    4.42, achieved + 0.003,
    f"realistic ceiling = {achieved:.3f}",
    ha="right", va="bottom", fontsize=8, style="italic",
)

# Annotations
for x, v in [(0, 1.0), (4, our_best_cv)]:
    ax.text(x, v + 0.0016, f"{v:.3f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold")
ax.text(
    1, (noise_ceiling + 1.0) / 2, f"-{label_noise:.3f}",
    ha="center", va="center", color="white", fontsize=10, fontweight="bold",
)
ax.text(
    2, (naive_floor + noise_ceiling) / 2, f"-{miss_cost:.3f}",
    ha="center", va="center", color="white", fontsize=10, fontweight="bold",
)
ax.text(
    3, (naive_floor + achieved) / 2, f"+{proxy_recovery:.3f}",
    ha="center", va="center", color="white", fontsize=9, fontweight="bold",
)

ax.set_xticks(range(5))
ax.set_xticklabels([
    "perfect", "label\nnoise", "missingness\n(mostly stress)",
    "proxy\nrecovery", "our best\n~0.907",
])
ax.set_ylim(0.89, 1.006)
ax.set_ylabel("balanced accuracy")
ax.set_title("The ceiling: two hard walls and one thin recoverable margin")
plt.tight_layout()
plt.show()

## 6. Our Best Submission vs. the Ceiling

We run our full LightGBM pipeline (5-fold stratified CV) to get an honest OOF score,
then compare it against the estimated ceiling.

In [ ]:
def preprocess(df, *, fit=True, num_medians=None, cat_modes=None):
    df = df.copy()
    if fit:
        num_medians = {}
        cat_modes = {}
        for col in NUM_COLS:
            if col in df.columns:
                med = df[col].median()
                num_medians[col] = med
                df[col] = df[col].fillna(med)
        for col in CAT_COLS:
            if col in df.columns:
                mode_val = df[col].mode()
                mode = mode_val.iloc[0] if len(mode_val) > 0 else "missing"
                cat_modes[col] = mode
                df[col] = df[col].fillna(mode)
    else:
        for col in NUM_COLS:
            if col in df.columns:
                df[col] = df[col].fillna(num_medians.get(col, 0))
        for col in CAT_COLS:
            if col in df.columns:
                df[col] = df[col].fillna(cat_modes.get(col, "missing"))
    return df, num_medians, cat_modes


def encode_and_featurize(df, *, fit=True, encoders=None):
    df = df.copy()
    if fit:
        encoders = {}
        for col in CAT_COLS:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
    else:
        for col in CAT_COLS:
            if col in df.columns and col in encoders:
                le = encoders[col]
                unseen = ~df[col].astype(str).isin(le.classes_)
                df.loc[unseen, col] = le.classes_[0]
                df[col] = le.transform(df[col].astype(str))

    if "bmi" in df.columns and "exercise_duration" in df.columns:
        df["bmi_exercise_interaction"] = df["bmi"] * df["exercise_duration"]
    if "step_count" in df.columns and "calorie_expenditure" in df.columns:
        df["efficiency_ratio"] = df["calorie_expenditure"] / (df["step_count"] + 1)
    if "heart_rate" in df.columns and "bmi" in df.columns:
        df["heart_bmi_ratio"] = df["heart_rate"] / (df["bmi"] + 1e-8)
    return df, encoders


# Load & preprocess
train_pp, num_medians, cat_modes = preprocess(train, fit=True)
test_pp, _, _ = preprocess(test, fit=False, num_medians=num_medians, cat_modes=cat_modes)

train_pp, encoders = encode_and_featurize(train_pp, fit=True)
test_pp, _ = encode_and_featurize(test_pp, fit=False, encoders=encoders)

feature_cols = [c for c in train_pp.columns if c not in ("id", TARGET)]
X = train_pp[feature_cols].copy()
y = LabelEncoder().fit_transform(train_pp[TARGET])
X_test = test_pp[feature_cols].copy()

for c in X.columns:
    if c not in X_test.columns:
        X_test[c] = 0
X_test = X_test[X.columns]

for col in X.select_dtypes("float64").columns:
    X[col] = X[col].astype("float32")
for col in X_test.select_dtypes("float64").columns:
    X_test[col] = X_test[col].astype("float32")

print(f"Features: {len(feature_cols)}")
print(f"X: {X.shape}  |  y distribution: {pd.Series(y).value_counts().to_dict()}")

### 6.1  Stratified 5-fold LightGBM

In [ ]:
lgbm_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_proba = np.zeros((len(X), 3), dtype=np.float32)
oof_labels = np.zeros(len(X), dtype=np.int32)
test_proba = np.zeros((len(X_test), 3), dtype=np.float32)
scores = []
from tqdm.auto import tqdm

for fold, (trn_idx, val_idx) in tqdm(
    enumerate(cv.split(X, y)), total=N_FOLDS, desc="CV fold", unit="fold"
):
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    model = LGBMClassifier(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="multi_logloss",
        callbacks=[log_evaluation(0)],
    )

    val_proba = model.predict_proba(X_val)
    oof_proba[val_idx] = val_proba
    oof_labels[val_idx] = val_proba.argmax(axis=1)
    test_proba += model.predict_proba(X_test) / N_FOLDS

    fold_ba = balanced_accuracy_score(y_val, oof_labels[val_idx])
    scores.append(fold_ba)
    tqdm.write(f"  Fold {fold + 1}: Balanced Acc = {fold_ba:.4f}")

oof_ba = balanced_accuracy_score(y, oof_labels)
tqdm.write(f"\n{'=' * 60}")
tqdm.write(f"  Our CV Results: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
tqdm.write(f"  Our OOF Balanced Accuracy: {oof_ba:.4f}")
tqdm.write(f"{'=' * 60}")

## 7. Headroom Analysis

How much room for improvement actually exists?

In [ ]:
# Recompute ceiling figures for final display
noise_ceiling = optimal_ba(train[train[RULE].notna().all(axis=1)])
naive_floor = optimal_ba(train)
realistic_ceiling = 0.951  # public LB top

label_noise = 1.0 - noise_ceiling
miss_cost = noise_ceiling - naive_floor
proxy_recovery = realistic_ceiling - naive_floor

print("=" * 60)
print("  HEADROOM ANALYSIS")
print("=" * 60)
print(f"  {'Metric':30s} {'Value':>10s}")
print(f"  {'-'*30} {'-'*10}")
print(f"  {'Perfect score':30s} {1.0:>10.4f}")
print(f"  {'Label-noise ceiling (complete data)':30s} {noise_ceiling:>10.4f}")
print(f"  {'Naive missingness floor':30s} {naive_floor:>10.4f}")
print(f"  {'Realistic ceiling (public LB top)':30s} {realistic_ceiling:>10.4f}")
print(f"  {'Our LightGBM OOF':30s} {oof_ba:>10.4f}")
print()
print(f"  {'Gap: perfect → ceiling':30s} {1.0 - realistic_ceiling:>10.4f}")
print(f"  {'Gap: our OOF → ceiling':30s} {realistic_ceiling - oof_ba:>10.4f}")
print(f"  {'  - of which is irreducible noise':30s} {label_noise:>10.4f}")
print(f"  {'  - truly exploitable gap':30s} {realistic_ceiling - oof_ba - label_noise:>10.4f}")
print()
print(f"  {'Our current position vs field:':30s}")
print(f"  {'  Baseline OOF':30s} {oof_ba:>10.4f}")
print(f"  {'  Top public LB':30s} {realistic_ceiling:>10.4f}")
print(f"  {'  Room to improve':30s} {realistic_ceiling - oof_ba:>10.4f}")

### 7.1  What the gap means

| Component | Value | Description |
|-----------|-------|-------------|
| Label noise | ~0.033 | ~1% injected label flips near sleep thresholds → ~3.3% BA loss |
| Missingness (recoverable) | ~0.017 | Missing rule features on ~26% of rows; stress hardest |
| Proxy recovery potential | ~0.010 | What good models claw back via correlated features |
| **Our current gap to ceiling** | **~0.044** | Room from 0.907 to 0.951 |
| **Truly exploitable** | **~0.011** | After subtracting irreducible noise |

### How to close the gap

1. **Better imputation** for `sleep_duration` and `physical_activity_level` (proxy features)
2. **Threshold tuning** for balanced accuracy (the optimal decision boundary is not at 0.5)
3. **Ensemble diversity** — XGBoost + CatBoost + LightGBM stack
4. **Target encoding** for categorical features
5. **Hyperparameter optimization** with Optuna

But realistically, **~0.951 is the practical ceiling** — the top LB is already there,
and ~0.033 is lost to irreducible label noise.

## 8. Summary

**The plateau is ~0.951 — and we are at ~0.907.**

The decomposition shows:
- **Label noise** (0.033) — hard ceiling from ~1% injected label flips. Nobody passes it.
- **Missingness** (0.017) — dominated by `stress_level` (12% missing), which is the only
  **unrecoverable** feature because it's independent noise.
- **Proxy recovery** (~0.010) — what a tuned model claws back via `step_count` → activity,
  `sleep_quality` → sleep bin.
- **Our exploitable gap** (~0.011) — the true headroom after subtracting noise.

**Key insight**: The 0.033 of label noise means even a perfect model with complete data
can't exceed ~0.967. Adding realistic missingness drops the floor to ~0.941, and proxy
recovery lifts achievable performance to ~0.951. Our 0.907 baseline has ~0.044 of runway
to the ceiling, of which only ~0.011 is actually recoverable (the rest is noise).

**Recommendation**: Focus on imputation strategies for the rule features and
threshold calibration for balanced accuracy — diminishing returns set in quickly.

In [ ]:
print("✓ Score ceiling analysis complete")
print(f"  Our best OOF: {oof_ba:.4f}")
print(f"  Realistic ceiling: {realistic_ceiling:.4f}")
print(f"  Exploitable gap: {realistic_ceiling - oof_ba - label_noise:.4f}")